# SINALA — Notebook Colab autocontido e reproduzível

Este notebook **não clona o repositório** e não depende dos módulos locais do projeto. Todo o código necessário para download, extração, treinamento, avaliação e demonstração está nas células abaixo.

Datasets baixados diretamente pelo Colab:
- Alfabeto A–Z via Kaggle (`heitorccf/librasign`);
- Saudações multi-fonte via V-Librasil, Acessibilidade Brasil e Spread the Sign;
- MINDS-Libras via Zenodo, com sinalizadores configuráveis.

A webcam usa JavaScript do navegador, pois o Colab não libera `cv2.VideoCapture(0)` diretamente.

In [ ]:
%pip install -q mediapipe opencv-contrib-python kagglehub httpx loguru tenacity
import sys

print(sys.version)


In [ ]:
import base64
import csv
import hashlib
import json
import shutil
import urllib.parse
import urllib.request
import zipfile
from collections import deque
from pathlib import Path

import cv2
import kagglehub
import mediapipe as mp
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

ROOT = Path('/content/sinala')
DATA = ROOT / 'data'
ARTIFACTS = ROOT / 'artifacts'
MODEL_ASSET = ROOT / 'models' / 'hand_landmarker.task'
MINDS_SIGNERS = ['01','02','03','07','10','12']
DOWNLOAD_MINDS = True
SEQUENCE_LENGTH = 32
DEVICE = torch.device('cpu')
ROOT.mkdir(parents=True, exist_ok=True); ARTIFACTS.mkdir(exist_ok=True); MODEL_ASSET.parent.mkdir(exist_ok=True)
print('Workspace:', ROOT)


## 1. Download dos dados e do modelo MediaPipe

In [ ]:
HAND_MODEL_URL = 'https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task'
if not MODEL_ASSET.exists():
    urllib.request.urlretrieve(HAND_MODEL_URL, MODEL_ASSET)
print('MediaPipe model:', MODEL_ASSET, MODEL_ASSET.stat().st_size)

alphabet_dir = DATA / 'raw' / 'alphabet'; alphabet_dir.mkdir(parents=True, exist_ok=True)
if len({p.stem for p in alphabet_dir.glob('*.csv')}) != 26:
    cache = Path(kagglehub.dataset_download('heitorccf/librasign')) / 'landmarks'
    for letter in 'ABCDEFGHIJKLMNOPQRSTUVWXYZ':
        shutil.copy2(cache / f'{letter}.csv', alphabet_dir / f'{letter}.csv')
print('Alphabet CSVs:', len(list(alphabet_dir.glob('*.csv'))))

GREETING_SOURCES = [
    ('acessibilidade_brasil','Oi','source_oi','http://www.acessibilidadebrasil.org.br/libras_3/public/media/palavras/videos/oiSm_Prog001.mp4'),
    ('acessibilidade_brasil','Nome','source_nome','http://www.acessibilidadebrasil.org.br/libras_3/public/media/palavras/videos/nomeSm_Prog001.mp4'),
    ('acessibilidade_brasil','Obrigado','source_obrigado_1','http://www.acessibilidadebrasil.org.br/libras_3/public/media/palavras/videos/obrigado1Sm_Prog001.mp4'),
    ('acessibilidade_brasil','Obrigado','source_obrigado_2','http://www.acessibilidadebrasil.org.br/libras_3/public/media/palavras/videos/obrigado2Sm_Prog001.mp4'),
    ('spread_the_sign','Oi','source_ola','https://media.spreadthesign.com/video/mp4/14/555228.mp4'),
]
VLIBRASIL_ANNOTATIONS = 'https://huggingface.co/datasets/ibmectech/v-librasil-raw/raw/main/annotations.csv'
VLIBRASIL_BASE = 'https://huggingface.co/datasets/ibmectech/v-librasil-raw/resolve/main/videos/'
GREETING_CLASSES = ('Ajudar','Cumprimento','Desculpa','Não','Nome','Obrigado','Oi','Por favor','Sim')
greeting_dir = DATA / 'raw' / 'greetings' / 'videos'; greeting_dir.mkdir(parents=True, exist_ok=True)
with urllib.request.urlopen(VLIBRASIL_ANNOTATIONS) as response:
    rows = csv.DictReader(response.read().decode().splitlines())
    sources = [('v_librasil', r['class'].strip(), r['user_id'].strip(), VLIBRASIL_BASE + urllib.parse.quote(r['video_name'].strip())) for r in rows if r['class'].strip() in GREETING_CLASSES]
sources += GREETING_SOURCES
for source, label, signer, url in sources:
    filename = f'{source}__{label}__{signer}__{Path(urllib.parse.urlparse(url).path).stem}.mp4'
    destination = greeting_dir / filename
    if not destination.exists():
        request = urllib.request.Request(url, headers={'User-Agent':'Mozilla/5.0'})
        with urllib.request.urlopen(request, timeout=60) as response, destination.open('wb') as output: output.write(response.read())
print('Greeting videos:', len(list(greeting_dir.glob('*.mp4'))))


In [ ]:
SIGNER_MD5 = {'01':'0259b309d2d5472f20d9842547d94a8c','02':'e2b8673264ba7fca4cfc071654a099e8','03':'18996a6bdc404e98a854bfd8a39d0268','07':'8e885f5cb6823744a583f206e4d2051c','10':'d6012f3e49e95d0cef7f6ec888743baa','12':'fff0716f21025cf8a3739c7e61d70db8'}
ZENODO = 'https://zenodo.org/records/2667329/files/Sinalizador{signer}.zip?download=1'
def md5(path):
    h=hashlib.md5()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(1024*1024),b''): h.update(chunk)
    return h.hexdigest()
def safe_extract(archive, destination):
    with zipfile.ZipFile(archive) as z:
        root=destination.resolve()
        for member in z.infolist():
            target=(destination/member.filename).resolve()
            if not str(target).startswith(str(root)+str(Path('/'))): raise ValueError(f'Unsafe member: {member.filename}')
        z.extractall(destination)

if DOWNLOAD_MINDS:
    zip_dir=DATA/'raw'/'zips'; zip_dir.mkdir(parents=True,exist_ok=True)
    for signer in MINDS_SIGNERS:
        archive=zip_dir/f'Sinalizador{signer}.zip'
        if not archive.exists() or md5(archive)!=SIGNER_MD5[signer]: urllib.request.urlretrieve(ZENODO.format(signer=signer), archive)
        print(signer, md5(archive))
    video_dir=DATA/'raw'/'videos'; video_dir.mkdir(parents=True,exist_ok=True)
    for signer in MINDS_SIGNERS: safe_extract(zip_dir/f'Sinalizador{signer}.zip', video_dir)
else: print('MINDS-Libras desativado')


## 2. Código de extração MediaPipe e features

In [ ]:
BaseOptions = mp.tasks.BaseOptions
Vision = mp.tasks.vision
options = Vision.HandLandmarkerOptions(base_options=BaseOptions(model_asset_path=str(MODEL_ASSET)), running_mode=Vision.RunningMode.IMAGE, num_hands=2, min_hand_detection_confidence=0.5, min_hand_presence_confidence=0.5, min_tracking_confidence=0.5)
landmarker = Vision.HandLandmarker.create_from_options(options)

def extract_frame(frame):
    rgb=cv2.cvtColor(frame,cv2.COLOR_BGR2RGB); image=mp.Image(image_format=mp.ImageFormat.SRGB,data=np.ascontiguousarray(rgb)); result=landmarker.detect(image)
    hands={}
    for i, points in enumerate(result.hand_landmarks or []):
        label=result.handedness[i][0].category_name.casefold() if i<len(result.handedness) and result.handedness[i] else ('left' if 'left' not in hands else 'right')
        hands[label]=np.asarray([[p.x,p.y,p.z] for p in points],dtype=np.float32)
    present=[p for p in hands.values() if len(p)]
    center=np.mean([p[0] for p in present],axis=0) if present else np.zeros(3,dtype=np.float32); scale=max(float(np.linalg.norm(np.concatenate(present)-center,axis=1).max()) if present else 1.0,1e-6)
    out=[]
    for label in ('left','right'):
        if label not in hands: out += [0.0]*64; continue
        normalized=(hands[label]-center)/scale; out += normalized.reshape(-1).tolist()+[1.0]
    return np.asarray(out,dtype=np.float32), result

PAIRWISE=[(i,j) for i in range(21) for j in range(i+1,21)]
TRIPLETS=((1,2,3),(2,3,4),(0,1,2),(0,5,6),(5,6,7),(6,7,8),(0,9,10),(9,10,11),(10,11,12),(0,13,14),(13,14,15),(14,15,16),(0,17,18),(17,18,19),(18,19,20))
def alphabet_features(coords):
    p=coords.reshape(21,3).astype('float32'); p=p-p[0]; scale=max(float(np.linalg.norm(p[9])),1e-6); p=p/scale
    distances=np.asarray([np.linalg.norm(p[i]-p[j]) for i,j in PAIRWISE],dtype='float32')
    angles=[]
    for a,b,c in TRIPLETS:
        u=p[a]-p[b]; v=p[c]-p[b]; den=max(float(np.linalg.norm(u)*np.linalg.norm(v)),1e-6); angles.append(np.arccos(np.clip(float(u@v)/den,-1,1)))
    return np.hstack([distances,np.asarray(angles,dtype='float32')]).astype('float32')

def sample_video(path, frames=32):
    cap=cv2.VideoCapture(str(path)); count=max(int(cap.get(cv2.CAP_PROP_FRAME_COUNT)),1); selected=np.linspace(0,count-1,frames,dtype=int); output=[]; i=0; target=0
    while target<len(selected):
        ok,frame=cap.read()
        if not ok: break
        if i==selected[target]: output.append(extract_frame(frame)[0]); target+=1
        i+=1
    cap.release()
    while len(output)<frames: output.append(output[-1] if output else np.zeros(128,dtype='float32'))
    return np.asarray(output[:frames],dtype='float32')


## 3. Treinamento autocontido dos modelos

In [ ]:
class AlphabetMLP(nn.Module):
    def __init__(self):
        super().__init__(); self.net=nn.Sequential(nn.Linear(225,256),nn.BatchNorm1d(256),nn.ReLU(),nn.Dropout(.25),nn.Linear(256,128),nn.BatchNorm1d(128),nn.ReLU(),nn.Dropout(.2),nn.Linear(128,26))
    def forward(self,x): return self.net(x)

def load_alphabet():
    X=[]; y=[]
    for idx,letter in enumerate('ABCDEFGHIJKLMNOPQRSTUVWXYZ'):
        with open(alphabet_dir/f'{letter}.csv') as f:
            for row in csv.reader(f):
                if row: X.append(alphabet_features(np.asarray(row,dtype='float32'))); y.append(idx)
    return torch.from_numpy(np.asarray(X)),torch.tensor(y)
X,y=load_alphabet(); model=AlphabetMLP(); opt=torch.optim.AdamW(model.parameters(),lr=1e-3,weight_decay=1e-4); loss_fn=nn.CrossEntropyLoss()
for epoch in range(10):
    order=torch.randperm(len(y))
    for start in range(0,len(y),64):
        batch=order[start:start+64]; opt.zero_grad(); loss=loss_fn(model(X[batch]),y[batch]); loss.backward(); opt.step()
torch.save({'state_dict':model.state_dict(),'in_features':225,'num_classes':26,'classes':list('ABCDEFGHIJKLMNOPQRSTUVWXYZ')},ARTIFACTS/'sinala_alphabet.pt')
print('alfabeto treinado:',len(y),'amostras')


In [ ]:
class GreetingGRU(nn.Module):
    def __init__(self,n_classes=9):
        super().__init__(); self.gru=nn.GRU(128,64,batch_first=True); self.drop=nn.Dropout(.2); self.head=nn.Linear(64,n_classes)
    def forward(self,x): _,h=self.gru(x); return self.head(self.drop(h[-1]))

glandmarks_dir = DATA / 'landmarks-greetings'; glandmarks_dir.mkdir(parents=True, exist_ok=True); gmanifest = glandmarks_dir / 'manifest.jsonl'
if not gmanifest.exists():
    gvideos = sorted((DATA / 'raw' / 'greetings' / 'videos').glob('*.mp4'))
    print(f'Extraindo landmarks de {len(gvideos)} vídeos de saudações...')
    g_recs = []
    for vp in gvideos:
        parts = vp.stem.split('__')
        src = parts[0] if len(parts) > 0 else 'unknown'
        lbl = parts[1] if len(parts) > 1 else 'unknown'
        sgn = parts[2] if len(parts) > 2 else '0'
        np_path = glandmarks_dir / f'{vp.stem}.npy'
        if not np_path.exists(): np.save(np_path, sample_video(vp))
        g_recs.append({'video_path': str(np_path), 'label': lbl, 'signer_id': sgn, 'source_id': src, 'frame_count': 32})
    gmanifest.write_text('\n'.join(json.dumps(r, ensure_ascii=False) for r in g_recs) + '\n')
    print(f'Extração concluída: {len(g_recs)} sequências salvas.')
grecords=[json.loads(x) for x in gmanifest.read_text().splitlines() if x.strip()]; classes=sorted({r['label'] for r in grecords}); c2i={c:i for i,c in enumerate(classes)}
X=[]; y=[]
for r in grecords:
    seq=np.load(r['video_path']).astype('float32'); X.append(seq); y.append(c2i[r['label']])
    for _ in range(40): X.append(seq+np.random.normal(0,.01,seq.shape).astype('float32')); y.append(c2i[r['label']])
X=torch.from_numpy(np.asarray(X)); y=torch.tensor(y); model=GreetingGRU(len(classes)); opt=torch.optim.AdamW(model.parameters(),lr=1e-3,weight_decay=1e-4); loss_fn=nn.CrossEntropyLoss(label_smoothing=.05); loader=DataLoader(TensorDataset(X,y),batch_size=16,shuffle=True)
for epoch in range(30):
    for bx,by in loader:
        opt.zero_grad(); loss=loss_fn(model(bx),by); loss.backward(); opt.step()
torch.save({'model_state_dict':model.state_dict(),'model_config':{'input_size':128,'hidden_size':64,'number_of_classes':len(classes),'dropout':.2},'class_to_index':c2i,'index_to_class':{str(i):c for c,i in c2i.items()},'metadata':{'feature_size':128,'sequence_length':32,'classes':classes}},ARTIFACTS/'sinala_greetings.pt')
print('saudações treinadas:',len(X),'sequências aumentadas; classes:',classes)


## 3.1 Extração e treino opcional do MINDS-Libras


In [ ]:
CLASSES_20 = "Acontecer,Aluno,Amarelo,América,Aproveitar,Bala,Banco,Banheiro,Barulho,Cinco,Conhecer,Espelho,Esquina,Filho,Maçã,Medo,Ruim,Sapo,Vacina,Vontade"
if DOWNLOAD_MINDS:
    MINDS_CLASSES = [c.strip() for c in CLASSES_20.split(",") if c.strip()]
    minds_root = DATA / "raw" / "videos"
    minds_landmarks = DATA / "landmarks-20"
    minds_landmarks.mkdir(parents=True, exist_ok=True)
    minds_records = []
    for signer in MINDS_SIGNERS:
        for video in sorted((minds_root / f"Sinalizador{signer}" / "Canon").glob("*.mp4")):
            prefix = video.stem.split("Sinalizador", 1)[0]
            label = prefix.lstrip("0123456789")
            if label not in MINDS_CLASSES:
                continue
            sequence_path = minds_landmarks / f"{signer}_{video.stem}.npy"
            if not sequence_path.exists():
                np.save(sequence_path, sample_video(video))
            minds_records.append({"video_path": str(sequence_path), "label": label, "signer_id": signer})
    (minds_landmarks / "manifest.jsonl").write_text("\n".join(json.dumps(r, ensure_ascii=False) for r in minds_records) + "\n")
    print("MINDS landmarks:", len(minds_records))
else:
    print("Extração MINDS desativada.")


In [ ]:
if DOWNLOAD_MINDS and 'minds_records' in locals() and minds_records:
    minds_labels = sorted({r["label"] for r in minds_records})
    minds_c2i = {label: i for i, label in enumerate(minds_labels)}
    train_rows = [r for r in minds_records if r["signer_id"] in {"01","02","03","07"}]
    val_rows = [r for r in minds_records if r["signer_id"] == "10"]
    test_rows = [r for r in minds_records if r["signer_id"] == "12"]
    def tensors(rows):
        if not rows:
            return torch.zeros((0, 32, 128), dtype=torch.float32), torch.zeros(0, dtype=torch.long)
        return torch.from_numpy(np.asarray([np.load(r["video_path"]) for r in rows], dtype="float32")), torch.tensor([minds_c2i[r["label"]] for r in rows])
    train_x, train_y = tensors(train_rows); val_x, val_y = tensors(val_rows); test_x, test_y = tensors(test_rows)
    if len(train_x) > 0:
        minds_model = GreetingGRU(len(minds_labels))
        minds_opt = torch.optim.AdamW(minds_model.parameters(), lr=1e-3, weight_decay=1e-4)
        minds_loss = nn.CrossEntropyLoss(label_smoothing=.05)
        minds_loader = DataLoader(TensorDataset(train_x, train_y), batch_size=16, shuffle=True)
        for epoch in range(30):
            for bx, by in minds_loader:
                minds_opt.zero_grad(); loss = minds_loss(minds_model(bx), by); loss.backward(); minds_opt.step()
        with torch.no_grad():
            val_accuracy = float((minds_model(val_x).argmax(1) == val_y).float().mean()) if len(val_x) > 0 else 0.0
            test_accuracy = float((minds_model(test_x).argmax(1) == test_y).float().mean()) if len(test_x) > 0 else 0.0
        torch.save({"model_state_dict": minds_model.state_dict(), "classes": minds_labels, "metadata": {"sequence_length": 32, "feature_size": 128}}, ARTIFACTS / "sinala_gru_minds_colab.pt")
        print("MINDS classes:", len(minds_labels), "val:", val_accuracy, "test:", test_accuracy)
else:
    print("Treino do MINDS não executado (sem amostras ou desativado).")

## 4. Demonstração com vídeos baixados

In [ ]:
from IPython.display import Video, display

videos=sorted((DATA/'raw'/'greetings'/'videos').glob('*.mp4'))
print('vídeos disponíveis:',len(videos))
if videos: display(Video(str(videos[0]),embed=True,width=640))

In [ ]:
video_greet = GreetingGRU(len(classes))
video_greet.load_state_dict(torch.load(ARTIFACTS / "sinala_greetings.pt", map_location="cpu")["model_state_dict"])
video_greet.eval()
video_path = next((path for path in videos if "__Oi__" in path.name), videos[0] if videos else None)
output_path = ARTIFACTS / "demo_video_unified.mp4"
if video_path:
    cap = cv2.VideoCapture(str(video_path))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    writer = cv2.VideoWriter(str(output_path), cv2.VideoWriter_fourcc(*"mp4v"), fps, (width, height))
    demo_temporal = deque(maxlen=32)
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        features, result = extract_frame(frame)
        demo_temporal.append(features)
        status = "AGUARDANDO SINAL"
        if result.hand_landmarks and len(demo_temporal) == 32:
            with torch.no_grad():
                probabilities = torch.softmax(video_greet(torch.from_numpy(np.asarray(demo_temporal)).unsqueeze(0)), 1)[0]
            confidence, index = probabilities.max(0)
            status = f"SINAL: {classes[int(index)]} ({float(confidence) * 100:.0f}%)"
        elif result.hand_landmarks:
            status = "CAPTURANDO JANELA TEMPORAL"
        else:
            demo_temporal.clear()
        for hand in result.hand_landmarks or []:
            points = [(int(point.x * width), int(point.y * height)) for point in hand]
            for x, y in points:
                cv2.circle(frame, (x, y), 3, (0, 82, 255), -1)
        cv2.putText(frame, status, (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)
        writer.write(cv2.flip(frame, 1))
    cap.release()
    writer.release()
    display(Video(str(output_path), embed=True, width=640))
else:
    print("Nenhum vídeo foi baixado.")



## 5. Demonstração com webcam do navegador

O botão abaixo solicita permissão de câmera ao navegador. Os frames são enviados para Python; `cv2.VideoCapture(0)` não é usado no Colab.

In [ ]:
import numpy as np
import base64
import time
import json
from google.colab.output import eval_js
from IPython.display import clear_output

# 1. Interface Web com Câmera Espelhada, Esqueleto e Controles de Texto
eval_js('''
(async () => {
    window.sinalaStopped = false;
    window.sinalaAction = null;
    const old = document.getElementById('sinala-root');
    if (old) old.remove();

    const root = document.createElement('div');
    root.id = 'sinala-root';
    root.style.cssText = 'position: relative; width: 640px; margin: 10px auto; font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, sans-serif; user-select: none;';

    root.innerHTML = `
        <div style="position: relative; width: 640px; height: 480px; border-radius: 12px; overflow: hidden; box-shadow: 0 12px 28px rgba(0,0,0,0.35); background: #0A192F;">
            <video id="sinala-video" autoplay playsinline width="640" height="480"
                   style="transform: scaleX(-1); -webkit-transform: scaleX(-1); width: 640px; height: 480px; object-fit: cover; display: block;"></video>
            <canvas id="sinala-canvas" width="640" height="480"
                    style="position: absolute; top: 0; left: 0; transform: scaleX(-1); -webkit-transform: scaleX(-1); pointer-events: none; width: 640px; height: 480px;"></canvas>
            
            <!-- Barra Superior HUD -->
            <div style="position: absolute; top: 12px; left: 14px; right: 14px; display: flex; justify-content: space-between; align-items: center; background: rgba(10, 25, 47, 0.88); backdrop-filter: blur(8px); padding: 10px 16px; border-radius: 8px; border: 1px solid rgba(0, 82, 255, 0.5); color: white;">
                <div style="display: flex; align-items: center; gap: 10px;">
                    <span id="sinala-badge" style="font-size: 24px; font-weight: 900; color: #00E5FF; min-width: 32px; text-align: center; font-family: monospace;">-</span>
                    <div>
                        <div id="sinala-status" style="font-size: 14px; font-weight: 700; color: #FFFFFF; letter-spacing: 0.5px;">POSICIONE A MÃO</div>
                        <div id="sinala-progress-bar" style="font-size: 11px; font-weight: 600; color: #00E5FF; letter-spacing: 2px;">[ □ □ □ □ □ □ ]</div>
                    </div>
                </div>
                <div style="text-align: right;">
                    <span style="font-size: 11px; font-weight: 700; color: #94A3B8; text-transform: uppercase; letter-spacing: 1px;">Modo Dactilologia</span>
                    <div style="font-size: 10px; color: #00E5FF; font-weight: 600;">26 LETRAS (A-Z)</div>
                </div>
            </div>

            <!-- Barra Inferior de Texto Acumulado -->
            <div style="position: absolute; bottom: 14px; left: 14px; right: 14px; background: rgba(10, 25, 47, 0.92); backdrop-filter: blur(10px); padding: 12px 18px; border-radius: 8px; border: 1px solid rgba(255, 255, 255, 0.2); color: white;">
                <div style="color: #94A3B8; font-size: 11px; text-transform: uppercase; font-weight: 700; letter-spacing: 1px; margin-bottom: 4px;">Palavra / Frase Soletrada:</div>
                <div id="sinala-phrase" style="color: #FFFFFF; font-size: 22px; font-weight: 800; min-height: 28px; font-family: 'SF Mono', Consolas, monospace; letter-spacing: 2px; word-break: break-all;">_</div>
            </div>
        </div>

        <!-- Barra de Botões Interativos -->
        <div style="margin-top: 12px; display: flex; justify-content: space-between; align-items: center; gap: 8px;">
            <div style="display: flex; gap: 8px;">
                <button id="btn-space" style="padding: 8px 16px; background: #0052FF; color: white; border: none; border-radius: 6px; cursor: pointer; font-weight: 700; font-size: 13px;">␣ Espaço</button>
                <button id="btn-back" style="padding: 8px 16px; background: #334155; color: white; border: none; border-radius: 6px; cursor: pointer; font-weight: 700; font-size: 13px;">⌫ Apagar</button>
                <button id="btn-clear" style="padding: 8px 16px; background: #475569; color: white; border: none; border-radius: 6px; cursor: pointer; font-weight: 700; font-size: 13px;">🗑 Limpar</button>
            </div>
            <button id="btn-stop" style="padding: 8px 18px; background: #EF4444; color: white; border: none; border-radius: 6px; cursor: pointer; font-weight: 700; font-size: 13px;">⏹ Parar Câmera</button>
        </div>
    `;
    document.body.appendChild(root);

    const video = root.querySelector('#sinala-video');
    const canvas = root.querySelector('#sinala-canvas');
    const ctx = canvas.getContext('2d');
    const badgeEl = root.querySelector('#sinala-badge');
    const statusEl = root.querySelector('#sinala-status');
    const progressEl = root.querySelector('#sinala-progress-bar');
    const phraseEl = root.querySelector('#sinala-phrase');

    root.querySelector('#btn-space').onclick = () => { window.sinalaAction = 'SPACE'; };
    root.querySelector('#btn-back').onclick = () => { window.sinalaAction = 'BACK'; };
    root.querySelector('#btn-clear').onclick = () => { window.sinalaAction = 'CLEAR'; };

    const stream = await navigator.mediaDevices.getUserMedia({video: {width: {ideal: 640}, height: {ideal: 480}}});
    video.srcObject = stream;
    await video.play();
    window.sinalaVideo = video;

    root.querySelector('#btn-stop').onclick = () => {
        stream.getTracks().forEach(t => t.stop());
        root.remove();
        window.sinalaStopped = true;
    };

    const HAND_CONNECTIONS = [
        [0,1],[1,2],[2,3],[3,4],
        [0,5],[5,6],[6,7],[7,8],
        [5,9],[9,10],[10,11],[11,12],
        [9,13],[13,14],[14,15],[15,16],
        [13,17],[17,18],[18,19],[19,20],[0,17]
    ];

    const captureCanvas = document.createElement('canvas');
    captureCanvas.width = 480;
    captureCanvas.height = 360;
    const captureCtx = captureCanvas.getContext('2d');

    window.sinalaStep = (hud) => {
        if (window.sinalaStopped || !video.videoWidth) return null;

        // Renderiza esqueleto e dados da última inferência
        ctx.clearRect(0, 0, 640, 480);
        if (hud) {
            badgeEl.textContent = hud.badge || '-';
            statusEl.textContent = hud.status || 'POSICIONE A MÃO';
            progressEl.textContent = hud.progress || '[ □ □ □ □ □ □ ]';
            phraseEl.textContent = (hud.phrase || '') + '_';

            if (hud.hands) {
                for (const pts of hud.hands) {
                    ctx.strokeStyle = '#00E5FF';
                    ctx.lineWidth = 3.5;
                    for (const [i, j] of HAND_CONNECTIONS) {
                        if (pts[i] && pts[j]) {
                            ctx.beginPath();
                            ctx.moveTo(pts[i][0] * 640, pts[i][1] * 480);
                            ctx.lineTo(pts[j][0] * 640, pts[j][1] * 480);
                            ctx.stroke();
                        }
                    }
                    for (const p of pts) {
                        ctx.fillStyle = '#0052FF';
                        ctx.beginPath();
                        ctx.arc(p[0] * 640, p[1] * 480, 5, 0, 2 * Math.PI);
                        ctx.fill();
                        ctx.strokeStyle = '#FFFFFF';
                        ctx.lineWidth = 2;
                        ctx.stroke();
                    }
                }
            }
        }

        const action = window.sinalaAction;
        window.sinalaAction = null;

        captureCtx.drawImage(video, 0, 0, 480, 360);
        return {
            frame: captureCanvas.toDataURL('image/jpeg', 0.65),
            action: action
        };
    };
})()
''')

# 2. Carrega o modelo de Dactilologia (MLP - 225 características invariantes)
landmarker = Vision.HandLandmarker.create_from_options(options)

alpha = AlphabetMLP()
alpha.load_state_dict(torch.load(ARTIFACTS / "sinala_alphabet.pt", map_location="cpu")["state_dict"])
alpha.eval()

text_buffer = []
current_candidate = None
confirm_count = 0
cooldown_letter = None
cooldown_frames = 0

REQUIRED_FRAMES = 6  # ~0.3s segurando a mão na letra para digitar
hud_state = {
    "badge": "-",
    "status": "POSICIONE A MÃO NA CÂMERA",
    "progress": "[ □ □ □ □ □ □ ]",
    "phrase": "",
    "hands": []
}

print(" Câmera de Dactilologia iniciada!")
print("Dica: Segure a letra parada por meio segundo para ela ser digitada.")

try:
    while True:
        res = eval_js(f"window.sinalaStep({json.dumps(hud_state)})")
        if res is None:
            break

        # Processa botões da interface
        action = res.get("action")
        if action == "SPACE":
            text_buffer.append(" ")
        elif action == "BACK":
            if text_buffer:
                text_buffer.pop()
        elif action == "CLEAR":
            text_buffer.clear()

        raw_frame = res.get("frame", "")
        if not raw_frame or "," not in raw_frame:
            time.sleep(0.02)
            continue

        b64 = raw_frame.split(",")[1]
        frame_bytes = base64.b64decode(b64)
        frame = cv2.imdecode(np.frombuffer(frame_bytes, np.uint8), cv2.IMREAD_COLOR)
        if frame is None:
            continue

        features, result = extract_frame(frame)

        all_hands_pts = []
        badge = "-"
        status = "NENHUMA MÃO DETECTADA"
        progress = "[ □ □ □ □ □ □ ]"

        if result.hand_landmarks:
            for hand_pts in result.hand_landmarks:
                all_hands_pts.append([[float(p.x), float(p.y)] for p in hand_pts])

            hand = result.hand_landmarks[0]
            pts = np.asarray([[p.x, p.y, p.z] for p in hand], dtype=np.float32)
            geom_feats = alphabet_features(pts)

            with torch.no_grad():
                logits = alpha(torch.from_numpy(geom_feats).unsqueeze(0))[0]
                probs = torch.softmax(logits, 0)

            top_p, top_i = torch.topk(probs, 2)
            conf = float(top_p[0])
            margin = float(top_p[0] - top_p[1])
            letter = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"[int(top_i[0])]

            badge = letter

            # Limiar de confiança calibrado para uso interativo
            if conf >= 0.35 and margin >= 0.08:
                status = f"DETECTANDO: {letter} ({conf * 100:.0f}%)"

                # Se acabou de digitar essa mesma letra, exige resfriamento ou mudança de postura
                if cooldown_frames > 0 and letter == cooldown_letter:
                    cooldown_frames -= 1
                    status = f"LETRA '{letter}' CONFIRMADA (mude ou afaste a mão)"
                    progress = "[ ■ ■ ■ ■ ■ ■ ]"
                else:
                    if letter == current_candidate:
                        confirm_count += 1
                    else:
                        current_candidate = letter
                        confirm_count = 1

                    # Barra de progresso visual (1 a 6)
                    filled = min(confirm_count, REQUIRED_FRAMES)
                    progress = f"[ {'■ ' * filled}{'□ ' * (REQUIRED_FRAMES - filled)}]"

                    # Confirma e digita a letra
                    if confirm_count >= REQUIRED_FRAMES:
                        text_buffer.append(letter)
                        cooldown_letter = letter
                        cooldown_frames = 12  # pausa antes de repetir a mesma letra
                        confirm_count = 0
                        current_candidate = None
                        status = f"DIGITADO: '{letter}'"
                        progress = "[ ■ ■ ■ ■ ■ ■ ]"
            else:
                current_candidate = None
                confirm_count = 0
                status = "AJUSTANDO POSTURA DA MÃO"
                progress = "[ □ □ □ □ □ □ ]"
        else:
            current_candidate = None
            confirm_count = 0
            cooldown_letter = None
            cooldown_frames = 0

        hud_state = {
            "badge": badge,
            "status": status,
            "progress": progress,
            "phrase": "".join(text_buffer),
            "hands": all_hands_pts
        }

finally:
    eval_js("if(window.sinalaVideo&&window.sinalaVideo.srcObject){window.sinalaVideo.srcObject.getTracks().forEach(t=>t.stop());const c=document.getElementById('sinala-root');if(c)c.remove();}")
    landmarker.close()
    print("Demonstração de Dactilologia encerrada.")


## Verificação

O notebook mantém o código completo nas células e não depende de `src/sinala` ou de um clone Git. Para repetir uma execução, use **Runtime > Run all** e conceda a permissão da câmera apenas na célula de demonstração.